In [2]:
import time
import random
import pandas as pd

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager


# TAMILNADU DISTRICTS

districts = [
    "Ariyalur",
    "Chengalpattu",
    "Chennai",
    "Coimbatore",
    "Cuddalore",
    "Dharmapuri",
    "Dindigul",
    "Erode",
    "Kallakurichi",
    "Kanchipuram",
    "Kanyakumari",
    "Karur",
    "Krishnagiri",
    "Madurai",
    "Mayiladuthurai",
    "Nagapattinam",
    "Namakkal",
    "Nilgiris",
    "Perambalur",
    "Pudukkottai",
    "Ramanathapuram",
    "Ranipet",
    "Salem",
    "Sivaganga",
    "Tenkasi",
    "Thanjavur",
    "Theni",
    "Thoothukudi",
    "Tiruchirappalli",
    "Tirunelveli",
    "Tirupathur",
    "Tiruppur",
    "Tiruvallur",
    "Tiruvannamalai",
    "Tiruvarur",
    "Vellore",
    "Viluppuram",
    "Virudhunagar"
]


# SEARCH TYPES

search_types = [
    "schools"
]

# CHROME OPTIONS

options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_argument("--disable-notifications")

# START DRIVER

driver = webdriver.Chrome(

    service=Service(
        ChromeDriverManager().install()
    ),

    options=options
)


# OPEN GOOGLE MAPS

driver.get("https://www.google.com/maps")

print("Opening Google Maps...")

time.sleep(15)

print("Google Maps Loaded")


# STORAGE


all_data = []

visited_links = set()


# SEARCH BOX FUNCTION


def get_search_box():

    selectors = [

        '//input[@id="searchboxinput"]',
        '//input[contains(@placeholder,"Search")]',
        '//input[contains(@aria-label,"Search")]',
        '//input'

    ]

    for selector in selectors:

        try:

            box = driver.find_element(
                By.XPATH,
                selector
            )

            return box

        except:
            pass

    return None

# EXTRACT SCHOOL DATA

def extract_school_data():

    try:

        # SCHOOL NAME
        try:
            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text
        except:
            name = ""

        # ADDRESS
        try:
            address = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"address")]'
            ).text
        except:
            address = ""

        # PHONE
        try:
            phone = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"phone")]'
            ).text
        except:
            phone = ""

        # WEBSITE
        try:
            website = driver.find_element(
                By.XPATH,
                '//a[contains(@data-item-id,"authority")]'
            ).get_attribute("href")
        except:
            website = ""

        # RATING
        try:
            rating = driver.find_element(
                By.XPATH,
                '//div[@role="img"]'
            ).get_attribute("aria-label")
        except:
            rating = ""

        return {

            "School Name": name,
            "Address": address,
            "Phone": phone,
            "Website": website,
            "Rating": rating,
            "Google Maps URL": driver.current_url

        }

    except:

        return None


# START EXTRACTION

for district in districts:

    for search_type in search_types:

        try:

            query = (
                f"{search_type} "
                f"in {district} Tamil Nadu"
            )

            print(f"\nSearching: {query}")

   
            # SEARCH

            search_box = get_search_box()

            if not search_box:

                print("Search Box Not Found")

                continue

            ActionChains(driver).move_to_element(
                search_box
            ).click().perform()

            time.sleep(1)

            search_box.send_keys(
                Keys.CONTROL + "a"
            )

            time.sleep(1)

            search_box.send_keys(Keys.DELETE)

            time.sleep(1)

            # TYPE SLOWLY
            for char in query:

                search_box.send_keys(char)

                time.sleep(0.03)

            search_box.send_keys(Keys.ENTER)

            print("Search Submitted")

            time.sleep(8)

            # SCROLL RESULTS

            try:

                scrollable_div = driver.find_element(
                    By.XPATH,
                    '//div[@role="feed"]'
                )

                previous_count = 0

                stagnant = 0

                for i in range(100):

                    driver.execute_script(
                        """
                        arguments[0].scrollTop =
                        arguments[0].scrollHeight
                        """,
                        scrollable_div
                    )

                    time.sleep(4)

                    listings = driver.find_elements(
                        By.XPATH,
                        '//a[contains(@href,"/maps/place/")]'
                    )

                    current_count = len(listings)

                    print(
                        f"{district} | "
                        f"{search_type} | "
                        f"Scroll {i+1} -> "
                        f"{current_count}"
                    )

                    if current_count == previous_count:

                        stagnant += 1

                    else:

                        stagnant = 0

                    if stagnant >= 4:

                        break

                    previous_count = current_count

            except Exception as e:

                print(f"Scrolling Error: {e}")

            # GET LINK

            listings = driver.find_elements(
                By.XPATH,
                '//a[contains(@href,"/maps/place/")]'
            )

            links = []

            for listing in listings:

                try:

                    link = listing.get_attribute("href")

                    if (
                        link
                        and link not in visited_links
                    ):

                        links.append(link)

                        visited_links.add(link)

                except:
                    pass

            print(f"Collected {len(links)} links")

            # VISIT EACH SCHOOL

            for index, link in enumerate(links):

                try:

                    print(
                        f"{district} | "
                        f"{index+1}/{len(links)}"
                    )

                    driver.get(link)

                    time.sleep(
                        random.uniform(2, 3)
                    )

                    data = extract_school_data()

                    if data:

                        data["District"] = district

                        data["Search Type"] = search_type

                        all_data.append(data)

                        print(data["School Name"])

   
                    # BACKUP SAVE

                    if len(all_data) % 100 == 0:

                        backup_df = pd.DataFrame(
                            all_data
                        )

                        backup_df.drop_duplicates(
                            subset=[
                                "School Name",
                                "Google Maps URL"
                            ],
                            inplace=True
                        )

                        backup_df.to_csv(
                            "tamilnadu_schools_backup.csv",
                            index=False
                        )

                        print("Backup Saved")

                except Exception as e:

                    print(
                        f"School Error: {e}"
                    )

        except Exception as e:

            print(
                f"District Error: {e}"
            )

# FINAL SAVE


df = pd.DataFrame(all_data)

df.drop_duplicates(

    subset=[
        "School Name",
        "Google Maps URL"
    ],

    inplace=True
)

# UNIQUE FILE NAME

filename = (
    f"tamilnadu_schools.csv"
)

df.to_csv(
    filename,
    index=False
)

print("\nExtraction Completed")

print(
    f"Total Schools Extracted: {len(df)}"
)

print(f"\nSaved File: {filename}")

driver.quit()

Opening Google Maps...
Google Maps Loaded

Searching: schools in Ariyalur Tamil Nadu
Search Submitted
Ariyalur | schools | Scroll 1 -> 14
Ariyalur | schools | Scroll 2 -> 20
Ariyalur | schools | Scroll 3 -> 27
Ariyalur | schools | Scroll 4 -> 34
Ariyalur | schools | Scroll 5 -> 40
Ariyalur | schools | Scroll 6 -> 47
Ariyalur | schools | Scroll 7 -> 54
Ariyalur | schools | Scroll 8 -> 60
Ariyalur | schools | Scroll 9 -> 67
Ariyalur | schools | Scroll 10 -> 74
Ariyalur | schools | Scroll 11 -> 78
Ariyalur | schools | Scroll 12 -> 78
Ariyalur | schools | Scroll 13 -> 78
Ariyalur | schools | Scroll 14 -> 78
Ariyalur | schools | Scroll 15 -> 78
Collected 78 links
Ariyalur | 1/78
Sabarmathi Vidhyalaya Senior Secondary School
Ariyalur | 2/78
Montfort Matric Hr Sec School
Ariyalur | 3/78
Golden Gates Global School ARIYALUR
Ariyalur | 4/78
St Mary's High School
Ariyalur | 5/78
Nirmala Girls Higher Secondary school Ariyalur
Ariyalur | 6/78
Kidzee International Pre-School Ariyalur
Ariyalur | 7/78